In [1]:
"""
End-to-end example: H, C, O on Cu(111) with NequIP calculator.

Demonstrates the three-step workflow:
1. Find sites – geometry only, no calculator.
2. Classify – group into symmetry-equivalent classes via graph isomorphism.
3. Optimize + check connectivity – relax each unique class and detect migrations.
"""

import numpy as np
import sys
import os
sys.path.insert(0, os.path.abspath("../.."))

from collections import Counter
from autokmc.structure import build_surface
from autokmc.surface import find_surface_atoms
from autokmc.graph import build_graph
from autokmc.default_sites import (
    find_sites_for_element,
    reduce_sites_by_isomorphism,
    optimise_site_positions,
)
from autokmc.adsorbate import (
    optimise_unique_sites,
    SiteOptResult,
    ConnectivityStatus,
)

In [2]:
# =============================================================================
# Load the Allegro/NequIP calculator
# =============================================================================

import torch
from nequip.ase import NequIPCalculator

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_MODEL_FILE = "asehcocuau.nequip.pt2" if _DEVICE == "cuda" else "cpuhcocuau.nequip.pth"
_MODEL_PATH = os.path.join(
    os.path.dirname(__file__) if "__file__" in dir() else ".", _MODEL_FILE
)
print(f"Device : {_DEVICE}")
print(f"Model  : {_MODEL_FILE}")


def make_calc():
    return NequIPCalculator.from_compiled_model(
        compile_path=_MODEL_PATH,
        device=_DEVICE,
    )


calc = make_calc()
print(f"Calculator ready: {calc.__class__.__name__}")


Device : cpu
Model  : cpuhcocuau.nequip.pth
Calculator ready: NequIPCalculator


/Users/bunting4/local/lib/python3.13/site-packages/nequip/ase/nequip_calculator.py:137: UserWarning: Trying to use model type names as chemical symbols; this may not be correct for your model (and may cause an error if model type names are not chemical symbols)! To avoid this warning, please provide `chemical_symbols` explicitly.
  warnings.warn(


In [ ]:
# =============================================================================
# Build example of oxygen adsorption to surface
# =============================================================================